# 01 — Preprocessing
Load → ICA → Filter (0.1–30 Hz) → Epoch → Save

In [1]:
# !pip install mne mne-icalabel python-picard pyriemann
# from google.colab import drive; drive.mount('/content/drive')
import os
# os.chdir('/content/drive/MyDrive/')  # colab
# os.chdir(r'C:\your\path')          # local
print(f"cwd: {os.getcwd()}")

cwd: C:\Users\joonp\Downloads\eeg_pipeline_final


In [2]:
import numpy as np, gc
from glob import glob
import os.path as op
import warnings; warnings.filterwarnings('ignore')
import mne; mne.set_log_level("WARNING")
from mne.preprocessing import ICA

try:
    from picard import picard; ICA_METHOD = 'picard'
except ImportError:
    ICA_METHOD = 'infomax'

try:
    from mne_icalabel import label_components; HAS_ICLABEL = True
except ImportError:
    HAS_ICLABEL = False

import importlib.util
spec = importlib.util.spec_from_file_location("cfg", "00_config.py")
cfg = importlib.util.module_from_spec(spec); spec.loader.exec_module(cfg)
for a in dir(cfg):
    if not a.startswith('_'): globals()[a] = getattr(cfg, a)

print(f"ICA: {ICA_METHOD}, ICLabel: {HAS_ICLABEL}")

ICA: infomax, ICLabel: True


In [3]:
# load raw
all_subjs = sorted([s.split(os.sep)[-1].split('_')[0] for s in glob(DATA_DIR + '/*.fif')])
raws = {}
for sub in all_subjs:
    fpath = glob(op.join(DATA_DIR, sub + '*_eeg.fif'))[0]
    raws[sub] = mne.io.read_raw_fif(fpath, preload=True, verbose=False)
np.save(CKPT['subjects'], all_subjs)
print(f"{len(all_subjs)} subjects loaded")

10 subjects loaded


In [4]:
# helpers
def clean_raw(raw):
    r = raw.copy()
    drop = [ch for ch in DROP_CHANNELS if ch in r.ch_names]
    if drop: r.drop_channels(drop)
    return r.pick_types(eeg=True, eog=False, stim=False, misc=False)

def ica_copy(raw_eeg):
    r = raw_eeg.copy()
    r.filter(l_freq=ICA_HIGHPASS, h_freq=None, verbose=False)
    r.resample(ICA_RESAMPLE, verbose=False)
    return r

def surrogate_eog(raw_ica):
    ch = raw_ica.info['ch_names']
    data, names, types = [], [], []
    hv = hh = False
    if 'Fp1' in ch and 'Fp2' in ch:
        data.append((raw_ica.get_data(picks='Fp1')+raw_ica.get_data(picks='Fp2'))/2)
        names.append('VEOG'); types.append('eog'); hv=True
    if 'F8' in ch and 'F7' in ch:
        data.append(raw_ica.get_data(picks='F8')-raw_ica.get_data(picks='F7'))
        names.append('HEOG'); types.append('eog'); hh=True
    if not data: return raw_ica.copy(), False, False
    info = mne.create_info(names, raw_ica.info['sfreq'], ch_types=types)
    return raw_ica.copy().add_channels([mne.io.RawArray(np.vstack(data), info, verbose=False)], force_update_info=True), hv, hh

def find_eye_ics(ica, raw_ica, raw_surr, hv, hh):
    n = ica.n_components_
    votes = {i: 0 for i in range(n)}
    if hv:
        try:
            idx, _ = ica.find_bads_eog(raw_surr, ch_name='VEOG', threshold=EOG_THRESHOLD, verbose=False)
            for i in idx: votes[i] += 1
        except: pass
    if hh:
        try:
            idx, _ = ica.find_bads_eog(raw_surr, ch_name='HEOG', threshold=EOG_THRESHOLD, verbose=False)
            for i in idx: votes[i] += 1
        except: pass
    w = ica.get_components(); chs = ica.ch_names
    fr = [chs.index(c) for c in ['Fp1','Fp2','F7','F8'] if c in chs]
    po = [chs.index(c) for c in ['O1','O2','Oz','P7','P8'] if c in chs]
    if fr and po:
        for ic in range(n):
            f = np.mean(np.abs(w[fr,ic])); p = np.mean(np.abs(w[po,ic]))
            if (p>0 and f/p>TOPO_RATIO) or p==0: votes[ic] += 1
    if HAS_ICLABEL:
        try:
            labs = label_components(raw_ica, ica, method='iclabel')
            for i,l in enumerate(labs['labels']):
                if l in ['eye blink','eye']: votes[i] += 1
        except: pass
    return [ic for ic in range(n) if votes[ic] >= MIN_VOTES]

In [5]:
# run ICA + single filter path
os.makedirs(op.join(CHECKPOINT_DIR, 'epochs'), exist_ok=True)
ica_excluded = {}

for sub in all_subjs:
    raw_eeg = clean_raw(raws[sub])
    raw_ic = ica_copy(raw_eeg)
    n_comp = len(mne.pick_types(raw_eeg.info, eeg=True)) - 1

    ica = ICA(n_components=n_comp, method=ICA_METHOD, random_state=ICA_RANDOM_STATE, max_iter=ICA_MAX_ITER)
    ica.fit(raw_ic, picks='eeg', verbose=False)

    raw_surr, hv, hh = surrogate_eog(raw_ic)
    excl = find_eye_ics(ica, raw_ic, raw_surr, hv, hh)
    ica.exclude = excl
    ica_excluded[sub] = excl

    # single 0.1-30 Hz path
    raw_filt = raw_eeg.copy().filter(l_freq=ERP_HIGHPASS, h_freq=ERP_LOWPASS, verbose=False)
    ica.apply(raw_filt, verbose=False)
    ev, eid = mne.events_from_annotations(raw_filt, verbose=False)
    ep = mne.Epochs(raw_filt, ev, eid, tmin=EPOCH_TMIN, tmax=EPOCH_TMAX,
                    baseline=BASELINE, preload=True, verbose=False, picks='eeg')
    ep.save(op.join(CHECKPOINT_DIR, 'epochs', f'{sub}-epo.fif'), overwrite=True, verbose=False)

    print(f"  {sub}: {len(excl)} ICs removed, {len(ep)} epochs")
    del raw_eeg, raw_ic, raw_surr, raw_filt, ep, ica; gc.collect()

np.savez(CKPT['ica_summary'], subjects=all_subjs, excluded={s:np.array(e) for s,e in ica_excluded.items()})
del raws; gc.collect()
print("done")

  sub-010: 3 ICs removed, 366 epochs
  sub-011: 2 ICs removed, 366 epochs
  sub-012: 2 ICs removed, 366 epochs
  sub-013: 2 ICs removed, 366 epochs
  sub-014: 3 ICs removed, 366 epochs
  sub-015: 3 ICs removed, 366 epochs
  sub-016: 3 ICs removed, 366 epochs
  sub-019: 2 ICs removed, 366 epochs
  sub-020: 2 ICs removed, 366 epochs
  sub-021: 3 ICs removed, 366 epochs
done


In [6]:
# build classification arrays
all_subjs = np.load(CKPT['subjects'], allow_pickle=True).tolist()
all_X, all_y, all_g = [], [], []

for si, sub in enumerate(all_subjs):
    ep = mne.read_epochs(op.join(CHECKPOINT_DIR, 'epochs', f'{sub}-epo.fif'), verbose=False)
    ep = ep.crop(tmin=0.0, tmax=EPOCH_TMAX)
    for cond, val in [('target',1), ('nontarget',0)]:
        if cond == 'target':
            keys = [k for k in ep.event_id if 'target' in k and 'nontarget' not in k]
        else:
            keys = [k for k in ep.event_id if 'nontarget' in k]
        if keys:
            X = ep[keys].get_data()
            all_X.append(X); all_y.append(np.full(len(X), val)); all_g.append(np.full(len(X), si))
    del ep

X_ep = np.concatenate(all_X); y = np.concatenate(all_y); g = np.concatenate(all_g)
times = np.linspace(0, EPOCH_TMAX, X_ep.shape[2])

# features
def feats(X, times, wins, types):
    blocks, n_ch = [], X.shape[1]
    for t1,t2 in wins:
        i1,i2 = np.argmin(np.abs(times-t1)), max(np.argmin(np.abs(times-t2)),np.argmin(np.abs(times-t1))+1)
        w = X[:,:,i1:i2]
        if 'mean' in types: blocks.append(np.mean(w,axis=2))
        if 'peak' in types: blocks.append(np.max(np.abs(w),axis=2))
        if 'var' in types: blocks.append(np.var(w,axis=2))
    return np.hstack(blocks)

X_basic = feats(X_ep, times, ERP_WINDOWS, ('mean',))
X_rich = feats(X_ep, times, ERP_WINDOWS, ('mean','peak','var'))

np.savez_compressed(CKPT['clf_data'], X_epochs=X_ep, X_basic=X_basic, X_rich=X_rich, y=y, groups=g, times=times)
print(f"saved: epochs={X_ep.shape}, basic={X_basic.shape}, rich={X_rich.shape}")

saved: epochs=(3600, 32, 701), basic=(3600, 64), rich=(3600, 192)
